In [ ]:
# ==============================================================================
# PARALLEL BRIDGE: EQUIPMENT
# ==============================================================================
from helpers import preload_dependencies, setup_logger, write_gold_table
from datetime import datetime
from helpers.silver_transforms import build_equipment_bridges

logger = setup_logger("parallel_bridge_equipment")

batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Ensure bronze dependencies exist
from helpers import IncrementalPipeline
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)
preload_dependencies(
    pipeline,
    ["inventory_equipment", "equipment"],
    watermark_overrides={"inventory_equipment": None},
    business_key_overrides={"inventory_equipment": "id"},
)

inventory_equipment_bronze = spark.table("wheelie.bronze.inventory_equipment")
bridge_equipment_group_equipment, bridge_car_equipment = build_equipment_bridges(
    inventory_equipment_bronze
)
write_gold_table(bridge_equipment_group_equipment, "bridge_equipment_group_equipment", mode="overwrite")
write_gold_table(bridge_car_equipment, "bridge_car_equipment", mode="overwrite")

logger.info("equipment bridges rebuilt")
